# 05 · API and observability

Run top-to-bottom in a fresh project Python kernel. Cells tagged `live` contact external services and may incur model costs. See [source attribution](../ATTRIBUTION.md).

In [1]:
from pathlib import Path
import os, sys, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/accounts_server.py").exists())
sys.path.insert(0, str(ROOT))
# Fresh temporary data for this notebook; never reset the dashboard's accounts.
notebook_data = tempfile.TemporaryDirectory()
os.environ["MCP_DATA_DIR"] = notebook_data.name
os.environ["MARKET_MODE"] = "fixture"
os.environ["ENABLE_PUSH"] = "false"
print("Using isolated fixture data.")

Using isolated fixture data.


## Test the dashboard API without a running web server

In [2]:
from fastapi.testclient import TestClient
from backend.api import app
client = TestClient(app)
assert client.get("/api/market").json()["paper_trading"] is True
assert len(client.get("/api/traders").json()) == 4
client.get("/api/market").json()

{'source': 'fixture',
 'quote_type': 'synthetic',
 'is_market_open': True,
 'paper_trading': True}

## Inspect a portfolio and its activity

In [3]:
from backend.accounts import Account
account = Account.get("Value")
account.buy_shares("AAPL", 2, "API demo")
response = client.get("/api/traders/value")
assert response.status_code == 200
assert response.json()["holdings"][0]["quantity"] == 2
assert client.get("/api/traders/unknown").status_code == 404
logs = client.get("/api/traders/value/logs").json()
assert logs
logs

[{'datetime': '2026-10-08 16:26:42',
  'type': 'account',
  'message': 'Bought 2 of AAPL',
  'color': '#dd0000'},
 {'datetime': '2026-10-08 16:26:42',
  'type': 'account',
  'message': 'Retrieved account details',
  'color': '#dd0000'}]

## Open the dashboard

From the project root, run `MARKET_MODE=fixture uv run uvicorn backend.api:app --host 127.0.0.1 --port 8000`. In another terminal run `cd frontend && npm ci && npm run dev`. Open http://localhost:5174. Use the offline demo CLI first to populate the main project data; this notebook's temporary accounts are separate.